# Designing a protein binder, step by step 🧬

### A tour of our "receptor resurfacing" pipeline

Welcome! This notebook walks through the **basic idea** of the pipeline we use in the lab,
using **one real worked example**. You don't need to be an expert — each step has a short
explanation and a picture.

## The biology, in one paragraph

Plants have **immune receptors** that work like alarm systems: they recognise specific
proteins (**effectors**) that invading microbes inject into plant cells. Our receptor here is
**Pikp-1**, a rice immune receptor. Bolted onto it is a small **HMA domain** that acts as a
*bait* — it grabs the fungal effector and sets off the alarm. Pathogens fight back by
mutating their effectors so the bait can no longer grab them. **Our goal:** redesign
(*"resurface"*) the bait so it once again binds a hard-to-detect effector variant called
**AvrPikF**.

## How do you redesign a protein? Three AI tools, in a row

| Step | Tool | What it does | Everyday analogy |
|---|---|---|---|
| 1 | **RFDiffusion** | Invents a **new 3D shape** (backbone) for the bait around the target | Sculpting a new key blank to fit a lock |
| 2 | **ProteinMPNN** | Chooses the **amino-acid sequence** that will fold into that shape | Deciding what to build the key out of |
| 3 | **Boltz-2** | **Predicts** whether the designed protein actually **grabs the target** | Test-fitting the key in the lock |

```
  target + bait  ──►  RFDiffusion  ──►  ProteinMPNN  ──►  Boltz-2  ──►  Does it bind? ✅/❌
   (a structure)      (new shape)      (new sequence)    (a test)
```

We'll follow **one design that works** — a redesigned bait our full pipeline found, which
Boltz-2 predicts binds the effector correctly on the first try.

> ⏱️ **Note for whoever is running this:** the tools download AI models and run on a GPU.
> In Google Colab, choose **Runtime → Change runtime type → GPU** first. The setup cells
> take a while (several minutes) — it's a good idea to run them **before** the student sits down.


## Getting set up

The example data is **embedded directly in this notebook** — one starting structure and one
finished design — so there's nothing to download or upload. The next cell just writes those
files out and installs a small 3D viewer.


In [ ]:
# --- 3D viewer used throughout the notebook ---
!pip -q install py3Dmol

# --- Example data (embedded, so the notebook is fully standalone) -------------
import gzip, base64

_EMBEDDED = {
    "af3_pikp1_native_avrpikf_complex.pdb": "...",  # target + original bait
    "design_28.pdb": "H4sIAJvaWWoC/31dWa5luY3871XkBu6DRkr6zFco2AbKA6qqG/D+N9LioCNR90b5o1+nnS9SRwNFMoLUzz///c8f8p/448e/5o+//fbfHz/1j/Lf0ldNmf+fr9LD/FG+AhX+Y5h/Cl9j/M/PByL9+PHLz3eI+hVaF4g0ikLwL3+EmP/ULx9GUb5oTPT5t3Or88/pq5cOIOY/8e8fn0ZRchGIlGn+OX71RgCi6lz89uv/CkQyiPxV6vyVV/iKpckoRk0Lgv/vAUE6FzfE/JUWBGL0oqOgCCCazsU7RO3BRsG/Gr6oIoiuc3FDxDmKKHMxUhCINtCHDJuL//4hEHl/SIwyCgqV/9pXKfkzRAw2F28QuXcZRUgM0W1lPkFEm4sLInx1nkaei54FIsYCIJLNxRtE5l998e6MApEDGkVeZ+RfAlEeiMS/Mj+kE8O2OScdQJR1RhzE/Fdb1blogwSiVDSddZ2RGyLnpMes8HTSPKlgX0RaZ+SGGIX/9fQV5azMM9MCgGh+X1SDaF+p6tZqsQsEJbQi3e+LBTFNjtkL2eA/ylcJYHfG4ffFgqhzRYJANB7ohGgIgg/Avz9A8L/adBS8+3guEoIw2/mP334VCHogRu4ynSPoKJJYr08QZjtviPxFlORnZdvJo8pgOpPZzneIwFZr/hyyMvPPHWytZLbzHaLx4eZ/vQ7dHwNs8GS28/9+/iYQzSDmSoymH1RtTiKCMNv5DhF4485RBLYTDClm+BOE2c4Lgu1EGLoyI+uHJLDBk9nOd4jedC5aJpteBDH8vuh7FIw+N3pMZHcrWJEc/L5YEHNXBrUTsen13JC9yNHviw3RZT+0edyCnNjSwe7Mye+LDZFkg7dphvVKqsiC5+ztxTCIPD9g6Cj4MM9R1IggircXG2K0JtYrjKqj4F36EaJ6e7EgyleUuRjzmDWByA3YzkzeXmyIIo7RvM3kEpgGEBm+3NwZ4fvRPqRXvY5z1ispD7Qi3Z2RA2LUKjd866QQBY1iuDPyQMyNPYJez3xCX9PfQLazBHdGDgjixWQ/I0QZFaGTWsx2/vztp0CYu/aq6vFNnyvxos1RpACOWTHbeUOQWvAJMdi8zlEkdJsVs53vo6jiDEx7weaVp7cjCLOd7xCBBzj9zz50FLkBw1fMdv7z1z8FIj0QUe6NOa38V/jPI352XYvZzhtiXj2869gLZjsxIQYlAGG28x2iy+GeH0K8tdp0lIAbX8x2fvgQtokcEfCd+ermSH+CGMvXEtc15mcUozPENHy8uHMUpYEPqWH5Wh6imrmdbhr/lTmKlsF01rh8rXsUmfd+nB9AczSvodfzR4i0fC0PMYMJ+ZDpc/GdOSFiQhDZxWaxPKOQizCy7ezzrE9ffPvgHGEcEMXFZg/EvL2KTueYfuVrnlgqAUBUF5s9EPNX+ITyKOadyhCxFQBBLjY7IArf2LyopB8ScgcQZjt//iGua6zPnUqkH9KnYeNRjJwAhNnOC0Istk4nh7oM0QL6ELOdN8Q0t033RZmGT1Zke3wegsx23hDT0FVbka5ba2SwIhT9XNDzIUOuoiG+N4+i5vF5a1Hyc0HPh/CWms7xV2ULzRu8gsNO2c/Fhki8AiloHMKHPWcAUfxc0DMXYShE5P3w4uACxOxktvMXvZZje+YiiGM0P8CsVu/oQ8x2XhBzayX5lTH9zy73Sm0Iwmzn+yiquO3tq8koaF5F4LCT2c53iGFWq/IZYFuKbCeZ7fz5+98Eou8NznCRpskhm06wIs1s5w0x94VYqTrPShWIgMxvM9t5Q0zPn+dgjiKw2zbnQo/dJwiznTdENg+PLbnORUe2s5nt/OPX3wVibAgJ5+YHVLtHEvoQs503xLwQ5RpuNgq2Xmg6zXbeEHwRZr1Hgl7LDSWlmtnOd4jR22P4xN+o4LA3H7On8HxIYJs4R9G7jiIT8HKaj9lPCDF046uw8yhXE3Bdm4/ZH4isMRnbTrYb0+MbBfid3cfsxyhIQpiuwcS8V0YCvlb3fmeKzyiiZBZtP3B8kkA80r3feUAE9v7mBs/sP8wPicj77d7vfCA4U6DuWhUvJ53H7ILwfucB0WRrVTW7knFEo/B+Z0rHBlePL/EH8FwMNBfe73wgyIz+jI54A7MPjiLE7v3OB6KZ2z5HkXRf1ALCu+79zgdiRoZFM8/iqrw4GQGioj78vsjPKHImy4frKBranSP4fZF3go7U5ORi+yKAFRnR74sNEUu13dks+4pGkfy+WBDzFhvq5WTSjGNsYEVGdnnwVJ5F7cOuomTZ14Y+pLg8+AFRn9uMLGmJIKrLgx8Qg0/gXJFUNdcHD/sglwd/IDj1kQ2i68mFK+Jj9rSTlWQnVTa6kBzA8A0fsx8QejJn+J80gdsRrTB8zP5AdDW7E6KyGWYIkGmMwcfsD8TcF1khhOCZ13SvDUBctpMeiCjHipMQTfPh4ENiuGwnPal0klhsZRyny5IHgLhsJz3kRubF4sMuAcY87Al9yGU76WEmSlGIFDXfmUDWNYbq/IvUNjPB/9PcF8LOMERAoyDnXzwQMyoqusFr1RxwBNey2JFffnyCUIs9zbDkw8sZp14Q3fkXB0Tv6jRW3h/T36ggcSupVZ6LP/+uEH1TLMnOiI2itc/Jyri4oneIQJqIqXzh8b0SwdZaXNENEc3bnVvLIAgt6uKK3iG6OEpsJ7L6XiECCB+zp/FACF+2SFBOnRLY4NHH7A/EvHqipcbYAPLh7wjCx+wHRJH85jwbQrVMCLgiPmY/PiRX5VGzxGt9jgpNZ3NzkcPDp1JRiEA6ipjRonY3FwdElfTxDHVtFKmiUQw3FwdEGJo+HnwRsjkG6eO4uKIPEJZKb3JSSX3xjxDR3SM5PitCSXlUSdwye5fBYV9c0TtEkszzmI4SGUMB5mJxRTfEnM6gJKheRdOSA1Y3Lq7ohmCGinRFgt6piFuOiyv6+cd/BOLg2dnkMI/KbtuEKICSjYsruiGm8V/8SFJGk4BzEBdX9A6Ro44iDr3ZI5zOvvIXHoITt0axDNUeoGx8TMOfkXxsLd0PvRTjmMF05uDPyObZq3wAKeHD/AggvWKO/ozk57CHocy2XP4MMcBtlpM/IwfPnpWWJmO6E4FFzd525s2zC6Xyo6nnx2lDZDuzt50LYu7KnLqx/BYJZLAvsred+SDJOxndpOQGvNmzt53HKHpQuULmrCpHy8iC58t2bp5dyE8+I9VohfE5KRXzZTs3hMwB/5Qk9rQXFUFctnPz7D0qBPFAOTsfP2dRYrls54YIWT+kcA5P2LwCILxGKdMDUYatSNTsWmjjY8o0Fq9Rypuql0wjX0Giz6Hpe0UA4TVK+aDqjckULpEzSqMCCK9ROkZROFXKJ5Wn8TX3Ry8Aovq5aE92rVY97MpkzsVtZ1KqHhDk52JDiK/F0g2+S3kULsd3QjQ/F5uql+zJHEVuxo+4ZOUJ0f1c7HwnDbWdWTb4PLmEPuSynTtZ2UWuQEInSDY+naM4jlm9bOeGyJzTZas1Dz1zAj0jiMt2Hmy/OQUUjJlIGUBctnND9KRzQZEplqT3ykeIy3aOB6IZVU/dIE4OsR9b6+KKHoj5K1XNbxOKZW61M6HvIC7buan6kEze1JVuyjEAiMt2HlR9H6Y50BVJDUF421nC8yHFrmVe99e6mp7pTAeEt51l8+zJrFaQRMzQi/EjhLedxyg6qexN3DHmRyKAIG87HwgO84e6a2xD+KSiUZC3nSU+HyJUPTvOZEns8pk3i+Rt5wEhjDb7naQJ/RgDgPC284FIGqPPUSjDXeY6AvNL3nYeEEV8LL7hlar3RM8J4W1nSYciRhdVUqeSSv+sMo3kbecBkYTtnvuC9FoeaQAIbzsXhDhG6msV9uj4Qwq4EMnbzgNCUmEiRNRrWbbaRwhvO0s+zG8311X1F4ScxuZt5wORTDk3T6rQTWl6v8BRat52HhDU9UKUbCt7fAWNwtvOAyIIP0BfaZhGCQUTzdvOUg75m94jUe7UuU8ScNeat50PxLzZuyoqJf/9Mi3bRwhvOx+IOgPsqj5XU5lT7gjC286yU6Zl+eBZpzNktCI+31m2uzY4d8dpoDEsxwcc6ObznWX7ncKP8HQWTdzCREzz+c5yer/VnEdN0GXkQHef7zwgxOwy1RI0yAwFTGf3MXuhHQnIGWF1UNMNX0HWtfuYveyUaW8KIZlnzrqiqKj7mL3slOnIqmlsFLfC8iOEj9nLTpnWoGLdJiuSIScQ+2U7j0xjUo2rmFeeE0JzcdnO9sSpcm8w7dQ109iAXiv2y3YeEE1THzlpdi12NIrLdh5ZV/mArF4vG8AEcnyLK/rzv5ImLP0ZRa962IM5TANlXRdXdENk9SuY0eS/wrcaoBXi4oreRyG2kqexap2A5DM+Qqx85xuEivvpawgpGmacikZx2c5xbK0mo9APmBdkR3Nx2c4NEUhHkUVWPw97BisyLts5njMih5jv1Gh5T0KjuGznhtDF7KYeSxq7f4RoTvdbw3PYxWEW0W4ysgOtSHe63weiGwQnH1RtejiNF8Rwut8HgvlUDXVVBx7gSU2LK7oh+DqO5v0mXdSRAUR0Ob4aN8RQxa0WBa108ieI5HJ8D0S1ZCWfjSQrEkCOLz1c0QUx7QSRrYxlohP6kOJyfAeEDJBFVsKThK8SEITZzv/8/m+BSFuLYouqAdW8U8EGT4sruiHY+9XEbRX/c94rEUGY7bwh5gp000DbB8WMptNs5w0xB150DsS75cUF2vgUvDa+5i3yLxpMkMhkpzkGtTQpem183a5rYEdJwv2srA1gJlL02vgDQqm3ZGKaacWA35mi18YvCHbPJIPEN7pu8ACUDyl6bXwtz1xI8Qdb8qi2E+6L6LXxD0SyC5BdWB1FBbqcFL02/oBQQWK2kjUWZ4INHr02/viQUZpVAipDUQExnJ66IqXq65ZF1q6Vf8JI8NbqaC660xwcEDR0LkLRpCWezuE0B/XQd+aTemuajPgE8dQVvUFo4mU6SinarQa21lVXVLe+U/nUpIl8CfPABr/qihbEaxm6pIoptuRoOq+6ogOiDqsIrRavojNy1RU9EEvUz1qlaCkycEaS59lre0aRxVqVeUKrZRDA7kyeZz8gWl3CdqXgAqFReJ79gFCZ03SUSM1wAvR0Sp5nPyDUSs1daYE30iil5DVKdes7hYmKpBubNzhQgaTsNUoHRI+qy+nB8hjojGSvUTogksgU5gd09fwIzUX2GqUDQuVuXbOtUq7VAITXKNWd75RM45wLikrVI4olZa9ROiBUo0Ra+ixpAHAVZa9ReiCicsqsMh3VyvnA1speo1R31jUHXREpy5lWrKHDvrgik8vSzjQ28fBIBasSVIAzsriiG6JotCzCs2jcMtpaw+l+TwiJP7rqOjnMSwBicUXvEEovMdtfLdgEi7q4IruKaCcrJRXKQtWmJAcK79Liit4hxFam6TjHYlEz2J2LK7oh2F0bfM+o3Il1Ocg5WFzRO4QUOCROnWoGIYLAKi2uaEHsZCXx1CbL4fAGj2g6yc9FOqoh9UOkKJ13J6iSTYsreh+FVAynqNrGH44TuCC6n4sNIWnhFLVWgi9IdMwWV2QpEMqH+dU6gd7Jyn7B7lxc0TvE6KqBzk1lkQmoQNLiim6IbFX0LMZTiIpWZHFF7xCDtFohR2W4CfkXF1dER13RUD24sngB6i/SxRU9EBabTYhgUkCky0kXV3RA9GpWq+q+iOikXlzRARGiKuhybkYIolE0F6fSqitqJh1mJlMhAlBKpV1XdENIRokFzCZI7AVtreHi1AeCr6BuvpaOgkCCLu26Ig9RVbbAKtOoi1ob2Frk9Z1EzyhUpMupMRXVqMbxE4TXdz4Qde7KoVoliRonBMj9JvL6zgOiWyij0h62WmguvL7z+BAx/px0aHoVNeQ0bq5IrmVqe4NHTcxJH4w5CqT7TZsruiFG0YyS2EzOAaPDvrkiD1G08IUjgGEXIrJamyu6IRarm4deAqjgIS2uyFwU6g+EGv+moQ3n+lDMvriiG4JpSCuJt0w0EiSmxRW9j0J1WnMxu96po6NRJOdrPRBW6MBpIaMXCspfNO930ma4C+liisaZfyIvp3m/84CQhByfDWlqwqI8NBfe7zwgZOOynSjKDQRCEN7vPEcR1E5QGpZSB7uzNdcXpW16WpSUYnZ1dxZQM5Fad31RDghR0rIO3DpQdEIrMlxflAXBUVEoVo9o8QhQmaYeXF+UA4JI20Uk0R7MDY68nIsranFnlMjuUlODVJBKTxdX9EDMxRx6l1aTfqEMdLq4ogeCP0CLgXrW8I7Qol5c0TEKCfs5arauPRGZ34srajvfKVQ9JyGKiTTRtXxxRW2XJmnhdbKmN81qND9BeK7ogZgWvGt/A4kIOFpGVuviitqubkomRJR8pnTtAS5K9zx7y5tPbfohw6QcRGAUw/Ps7SxN0iIx8T+lDANBeJ697boiyZot8R2THMhFGZ5nP0bRZC44x6cQETkHF1fUtiyy8jTyZZCrFUx9li2kiytqWyIqrgl34zD1WAXtytLFFT0QHAmoaKIkUxXWBCC83/lAcF5LpaFLmNgIQXiNUquHaLfolTT0eq6gGD0Nr1F6IJhTHuZz6RnptQMIr1E6ILoJVMl6SjWgy8nBa5QOCGkgxxCjmXCgAQivUWr0yKiHtM7gG94UluOzLicHr1F6IGjeZqoKqkP9jJQDgPAapQdiTl+xDzFelTIahdcoPRBdU2PsPDZjNFMGEJftbA+ECpeLSTmKVXIsiHZAXLZzQTDlplXT8ld+5LNM64K4bKdBpKAlKOx/lmDmuAOIy3ZuiBF1RSSrJkmpASCOfOe31iCLb6A9Yvgn3x8vqbn6XBKf45Hv9BCVWAKYpSSe1YS1gxWJR77zhGAHmgQiF4Uo4LDneOQ7T4gsStuXVfbwTyTmzosr4lT6txZTGwTXpcrPplrXAsL+vLiiGyKZHHJGALkKBMoo5cUV3RBRtM8yJzYKdJvlxRV9gBCB6tydWcWZCWQaczxi9m+tCheIICVJ/CFN2okwe0cA4ojZNwTrOZkD4g+pjWRukAOd4xGznxBNOpzJKETrmjQH/AkiHTG7h2hDxbq8seUncF3z4orYjf/W8nYLdcfQD+F4ROSyoAQlL67ohqjCHcoGb00gkAokL67ohlC3XXepSoczKCLMiyt6hwhDf1WbmiT1yT9CHFzRt9bpr3SQzAFnW3VlYkBzcXBFJ0T5YmKPIRpl+Zkjgji4Ig/BN7kcerNeqNtCPrmiEyJLsZisSOJREKTqczp6gXxrw4GVv7BdyQ1JeRQoQZfz0QvEQ3CbGRG0U5RRoNr+nI9eIB6iy0kNUgYuFp0agDh6gVwQkm3lBi9DRlFATWbedUW6tXaasMRlM/WslIBGUXYux0MIz/4aUij1klpucEZ2XZGHYJ2viriZkeBRHD1iLgjauZwTIilfxtZL9seEGANANL8v6jGKoL1hZE6K1uB9hOh+X9RnFJVdEu6TQ3rcRkYrMvy+qJuSFR04iSZFzgrQa+WTK/KjkLCN1ehJbWcFBGAu0d8j9JAbsg94FFnv1g6ya7kkf49sql46Zr60AFutFoLI/h7ZEMlCGrUXSVMiHyGKv0c2hIhx2ddKept10N4wl0Oj9K1NOVYHIwlp2J3v5msB77ccGiUPoW0N81eytlSxIYhDo3RCRG1DJWF/0xMbC4A4NEonxFyBpt03un1ILcDjW1wRt6X61u4iSwVSVhNndZR8jdUBsbiiG8IIwJcmp8TLGRVAxN1f64SY0xh1FNqdpVvrx08QaffX8qOo1o1a3DF2WQIBCLOd//m7uGt9N0ESKY/InbRkbQzggy+u6Ibg+FSTUpW0y1dr6EPMdt4QZHUCQTsMcJevgiDMdt4QTXs88HEb2hXwaCdyQTQ3F+NQViYdhZas0VlcekF0Nxdj6ztFysP5TrnVpg0tCGK4uRhbZbpyfMEaQhXQ5Ssvruh9FN2ae4tE9FVO6fAF4W3niLsndtNe6RKTsRMJBEaZvO18IKJ2AWShkRnAjtw18rZzHCXxQWslRAHB7htImWbytnPEQ4kdNpsrnCIwv1SdvRi7Y7ue0KDiO6Hu0YeQsxdjl8RLl3Le4FStlgb4WtScvRhHSXwtu1MiaxDQzU7d2YsTQlr5WetgbkuF/AvyMfvITy5HUmEv68vH2hS0Is3H7A9Es+6h1s6O9RgoyGw+Zj8gJNkgDYyVJO+A4c7Nx+wPRNc2M+xnFN2lHXm/iytaW6s8o5DgUXZn15VBbvziim6IbvWHHJOp8qEOBFH9GdmjkBJFZmuEyYzWq/ATBPkzsiGkvy9vKdLm3qhHTG7N5S9GfRJ0wTqdCf3MbD/ywVt3+YsHomrxh5Bf1h8cNPLIbbj8xdjJymF5LWkAxHI4NBc9uPzFOFOmmj6WVPlflGnlxRWZizJop0yls0DV2jvpKgrOyOKK3iGEOnlZwvZHOPNaF0R2vtYDkbVRGF9BWXdpRht8cUU3RNJi0iXlYF0fyuX0o2/8t7bbslFE688otVUvLGDO/egbf0Ik5dWlV2Uww4fm4ugbf0KYIp/9Cgp2VkAiph994z2EdCnnCDFoc+8C+kDnxRWt6ezPzU72IcLOiboQZA4WV3RDsKRHvRuplRB1EPiQxRW9QwQJrIb9TMoNfIRIfl8cEEM7C0hAJUEngvAx+9juGklwaW1EXnbTf4TwMfsBUa0gX4qC+B5BCbrhY/ax/c4wNKumjCZ3NAKLOnzMPrb320nzWtLwh7twAHl9XlwRt4b91k5uBsHN0HgutOl71o4DHyH67nF7QmTRoEhuz4qOK2Ay8+KK3iGCNSxuTQ1gBGx/WVzRO4QGVl2jZJ5O0FiwLK5IXVcpvl8MN28ZbptMxXqnI4h0+uAbIhqfykLVxdYMAJFPH/yBeFkJCrcFqGTC1QggyumDnxB8BWlkqFdSA8mHsriiP3//j0Ckoye2HjMRrHLsntFcmO28IaaJGZqUitYOdRQ0CrOdN0QyCOugyR8CrqKyuKJ3CKqWDop6n2QgfyuLK1rTmY+2EXqPLHZX6OpPEIsruiGqVmpIasz2BaBYyuKKbgjuHkp2FXXN6QCrVRZX9A4xrCuLVLEIZQ/mYnFF3A71W7sdLk3japtsW6uCl4LK4opuiK7lN/xMg3VbaECJXRZXdEOM6SRWm8aicwEy0GVxRe8QpVtkKJHiHE1Ho2h+X9SHKyLL8UXjVWNDK9L9vqgPYyWCRMk0ah+lAfTgJQ6/L+rD3oViKxKtfTKazhT8vtgQ0n+RRZpJAytoctKhjf/W/pMGMYye1sNeYC1NSYc2/oSo+sSRNPAo9iHgjKRDG39CcM5XD3mxFRmAEyjp0MZ7iLIaeHTLvoLi0pJczC4xh0A0yySRSnrkjAB7kVzMviHY7BY9btU+CIS6JbmY/RyF9sKeroptsVbQorqY/YQYQ51GsY0v68bxEWL4uejPdCZr9i5P5r0IH7Mc/Fxsnr10fQRAikt9ju+CiH4uNs8+bC4KZ8/42Q5Qb1Zy8nOxITQqYl5Vr+cG+NSyuCJ14/XVKP2QaI8AiOr81eGrSWVxRTdEsU5nVatkma1B98jiim6IrL0IJSWmvYMqsp2LK3qHCLaxyUKbDJIPxXNFMYYtGAhqrairz5XhdDquaEMELRKTDGPfEcFHCMcVOQhLifVmTdOQl+O5og0RzeBFbVwsoQ1Ykd2DTraWPfOjYgmLhrJuMXiP7B50bxCh2UuEarUqyK6V3YPuhBDHuWmmcaWP0et7ZfeguyFkpjmJPRQCsXel1DOvFZ9nfpqKdTm4tH58iNwohc681oYgbeAhc2EUXGoAop15rXg8NlSM6KnVTizIxpfSz7zWCSHvY3I0JHajTnsRAITLd8bjpSBpnigJ26z3CWhSUKrLd8bjpSBpAMZqwqS8agaPAZTq8p0bItq7Iywpr9YhEdzs1eU7TwiRZvAZMYcJlaCUs67oW/sYr6LjYSK8lUpHx+ysKzoghAytxiEu0wMO+1lXdELEJxoSyy06TzSdh77Tj4LMv1CRZtFmWR8h2u6J/a0NmVdhfun2kpR+EHrVoCyu6IbgLtTqgxeysB8u6tg9sS8IyubG910R+AlicUXvH6IPwbL7rmnCgSz44oos7I+7c6b8T2xqQtwG8CNEcvmLA4Ks6WayerMI8lplcUU3RJH+jGoAVamPniQoiyt6h2jrRrfyPQISjrK4Im65860tslcux9hcbZPLnTTRXNDuHXRCkAYP7N2sYjHkri2u6H0UOekTWMFS6ajReVlc0Q1RxEESCKtbRl2Hy+KKVOYUn2d+qoS64meQfkhBPvjiit4h2A1jb0ceSGBOAM3F4oreIbgwX9QfVsON+v2WxRXdECQKWxmFVfSgtyHL+V7RtzYtf14KKio8sxZdqG98Od8r8hCqqGS5bLZedMBenO8VeYhiHzJWR7yKRkF+LnbKNIl0I1urhKYVXx8hmstrnc/8FBXTjK71Zn2gD+kur3VC2AZvST+EUDCxuKIbIosKXXqYZrK2dmAuFld0QySbi2TvCRAkAEv3Mfv5XlFSDYr861IUhCB8zJ7i0SbXOuwOayeCzkj3MfsJITd7sw4DTQPwjxA+Zk87cTvysuA6igp4s3K+V/St/fDX+2ZVA26ZA1kRcBWd7xV5CH0mrmrfTmlTBQ77+V7RAfGylz/E/yxWmgRs5/le0QkR9RkwyeV0E5ag6RzOB0+7/VBfioeSrboJLOrzXtEFwS7KsJqJYYUwYGs97xW9QcgjAOx3hmBd7MGKPO8VXRCcXVOIEK2PJ9oXI/szcigru7IzIt3gNpgoHTSKPyO7JF7oReYOo1axwBUZ1Z+RLRFtq3tos+JSdI8M8mekPFERme6iZu2XE4Aipozm52JXkks9u7gmwd4oRyvS/Vwc9eyhW2NrOp4J+wQx/FzUnT42iG7NN9GLMDUEPxd7FNrz2FQgrOsLCCK6XM565kcSL6p8kOcEuUq2I4jkcjmJnphdGoSJbGGoOQbTWUN2uZwHImptv2he7Zkf4DTWUFwu54GwztwiYynWk64BiOpis9SeUZAJEls0dw2E/fWsKzoh0vPsU0jVWnYNANFcbPZAZKuaDhqv8vsjEa1Id7HZMQrpZCYvVeoGzxDC8ewx7XIc4YbkLXJ7rwjEqTU6nn1DFN0XzCWmZq4sAQjHs5+jELksz4XdqRHtzuh49hNCn7aJ2uTmeT7uE0R2ud+0i4IkwH5ZYaU0hEKjKC73e0AEq5IN0SpDwZ1aY3W53wciaGMw6fZlHVUBY1UjudzvAZEsuKSaH5/8M4T3O/MuCupmL+R4rZdAPkJ4v/OB6Ko94cNuczEKgvB+5wlhVbLS3Y3LfYGkvCbvdx4fomF/tkcMA6SbavK2cz3z82raQ+q1oqJubao+QXjbeUJEe25UPL4OM9A1eduZd3WTpDyEN1MXBb1VX5O3nXlXNyWDyKsrYEBzYbbz7/+QqygfacJAO9vKdczI5Cyu6IbgLLxyy2ovhmqhP0KY7fwAEVQ8UewpkwpSY3VxRe8fouqgoa4Jq5GBwKimcer4Yt7JymavPQv/IU9WAIgcTh3fCaGPAAyDIEhu1BxPHd+GSFI5Loe+6PMdDfDsNadTx7chor1mMHSjMwTo5lSz9zvzLsepUQukpGqBIYADXbP3O/NOmbJWkW2o5K3kZRg0Cu93HhChdSN8xl82NanZ+50HxBjBKgCHvbMLttbiiizsz/tx9RS0kic/yQdg+BZX9A5B1TIH1q5sDPQhw+Uv8m6+WaQ+lf/1aEkqsLUWV3RDsMCIjOnOO2H3EcJp42PeL6NrvbK1j5AGLwjCaeNPCI6Wmd0VL4bzn8hd83VF8XiviBMvArEcaEJz4bTxJwR34+EPqeZnoMfJ6sUVPc/8ZMkYMFkuzqI0aUVz4bmivJOVwSA0lxPPlz8uCM8V5Z2sbCK26toKVMgvNArPFR2jqFIw15/yi4JuM19XFPPxrLkUlXbNorCKDO0LX1d0QnDNNkOMpN5OAfWp1dcVbQhtr877okVzVUD+ovq6oni8mjSSba1qT3kjp3FxRZZKX8/8MD8Sqmytbhr5jq6ixRXdEKywrVrjn4M1aUUQ1XECx6tJ+nJU1YaTLxPxfoQgxwmcDy+R1mJqy4QIn6qouwedTOfzzE+Ql2x5LlaMVhJake72xfHYUIjNILq1RwUX4u5Bd0Mk0vRxsz6NqOi47h50HiJKLxCBsEdxI8iiVHL17PF4bIhb6YgaJOkoAvI7ydWznxBM0es9om3s0CsolVw9+4ZgPSdp+XfVgoeGDB+5evYNUeVxspfUCxQjBNEoXD17LNvjG0GL0QNF6/KFRuHq2TcEiSZelDHBmqahrUWunv2E4AajLhsPGkJVcvXsG6JK9kTaApi7liDEcPmL9cwPK6RqsBYa+S/fT63ne0UeokuxcZT2AOJrAaq+nu8VnRDji0sIJKdjTyuiYrF6vle0IUQFQvZMXNaHPxPgiqqvK4plyyJb06p6oWQ5cYtOqq8rWhCiiBkqKZdKLm4QldB0urqiE0IoFSlG1/ZDGR12X1d0QqjmoKuKUDgBsMGb13eWulNjchGa2lQeHUJz4fWdB0Q1wZnE5D+wpLw2r+8sW5y5ZNTBGsgVZHK613c+EMVUpvNmX2/1gHd1q68rioUO+VuwoiDtfZzQdPq6og1RrXqavrI9cZMT+hBXV7QhSF/3ZVlk0GaLCRSLVV9XdI5C0j8SLVvKFFQr1F79XGxZpLRNFiW2tXxE/kUnPxdb36kqkKaqIJYxJDSdzc/FhlhzES1litqt1979XBwQKe4kBHu/aHf2y3b2B2JpGYOR5AFIe+q4bOcBEXVFKKv3m5HrOi7buVWmUhcgQkSFaKB+pI7Ldm6IanXsokPguBV5OYsrstRY2fpO5c2aFREGnEpfXNENwfrOZq13NHUaUQpkcUU3RH60KNLInFPqoBVTXVzRO4S8Qc65vtTNUULT6WoyY936Tm1S0C11GuwRkU8QriZzQ0Tb4PMSKMrWEFxUV5O5IYK+Cs9637rc+M8QFFxN5gkRjBgexeqWawMQPt+5nvmRBz616k16lf7FI9p0cUV15zubMFbDJMTcPgKNwuc7D4ho/XJECcPBBEjo08UVnRBJk1Kq7+T+FxFAVOeD1+3xVXGU7MkjznOBfUGLK3qHCEEfgJU+OUzNVgTRnA9ejz5KOZufoewu4lNpcUXvEKyE0TZlqmEL8EO87axHvjOn3diFRXgg4KbobWfdyUpu8fhar63xe6rgNqPobWc9Ws9H1eVk+xACb71R9LbzGEUlbTukNXgFkhsUfb5zPfMjh1sDq5CK9dvKAMLnOw+IYE3TaHXvAVlXij7feTw2VIZCSLqHtxpIE1L0+c7jyaOQ8s66cqyW0Vx4nv14bOjpn0TKDWQ4F55nr/V4UNzalTXVg2fgulL0PPsDwepjTZnK+7ocJQEXhZLn2c9RlKH3SVMvGFlwWlyRyevrfp89SB++pE/aiMQHQSRXJ3BA0NDwLpRhIn+wLxZXdEOQpY+TOgVshoFzQIsruiGUnVGCWEdRgVyWkvc76xZn9myNoEKynmNoFN7vfCCayJz0sKsNRR0SKXm/84EYFmRmIcfFADYE4f3OA2KJ8PjhA6FaKpqLy3buenauUpANbk2QUJMCypft7PvZ0aoQZCXQEUTLlC/buSGSJaW0pqZDMTfly3budzLTUFpBSPBXhyENZddHKdbjhcqhaULp4iSuCvoQ10fphAiW7yQrFkMJfcquj9IDIY9na5pwkHYwQrkcyq6P0obo0pdRCuiGFdxex+z/ATzJd010pgAA",  # a redesigned backbone
}
for _name, _b64 in _EMBEDDED.items():
    with open(_name, "wb") as _fh:
        _fh.write(gzip.decompress(base64.b64decode(_b64)))

with open("design_28_seq_1.fasta", "w") as _fh:
    _fh.write('''>receptor
GLKQKIVIKVAMEGNNCRSKAMALVASTGGVDKVYAVPGSDKITVVGYGIDPIKLISALRKKVGDAELLQVGGMKRL
>effector
AIDLSRERDPNFFDNADIPVPECFWFMFKNNVRQDAGTCYSSWKMDKKVGPNWVHIKSDDNCNLSGDFPPGWIVLGKKRPGF''')

# The three files we use for the worked example:
INPUT_COMPLEX = "af3_pikp1_native_avrpikf_complex.pdb"   # target + original bait
DESIGN_BACKBONE = "design_28.pdb"                        # a redesigned backbone
DESIGN_FASTA = "design_28_seq_1.fasta"                   # its designed sequence
print("✅ Example data ready (embedded — fully standalone).")


## Step 0 — The starting point 🔍

Here is the structure we start from. It contains **two proteins stuck together**:

- **Chain A — the bait (Pikp-1 HMA domain)**, in <span style="color:#3b82f6">blue</span>. This is
  the part we will redesign.
- **Chain C — the target (AvrPikF effector)**, in <span style="color:#f97316">orange</span>. This
  stays fixed — it's what we want the bait to grab.

The <span style="color:#ef4444">**red**</span> patch on the bait is the **surface we will
resurface**: we throw away those residues and let the AI invent a new shape there.
Spin the structure with your mouse.


In [ ]:
import py3Dmol

# The "contig" is our recipe telling RFDiffusion what to keep and what to redesign:
#   A1-32   : keep bait residues 1-32 exactly
#   10-20   : invent a brand-new stretch of 10-20 residues here  <-- redesigned
#   A46-72  : keep bait residues 46-72 exactly
#   6       : invent 6 more brand-new residues                    <-- redesigned
#   C       : keep the whole target (effector) fixed
CONTIG = "A1-32/10-20/A46-72/6-6 C"

# Residues of the ORIGINAL bait that get replaced by the new design:
REDESIGN_REGION = list(range(33, 46)) + list(range(73, 79))   # native A33-45 and A73-78

with open(INPUT_COMPLEX) as fh:
    pdb_str = fh.read()

view = py3Dmol.view(width=760, height=520)
view.addModel(pdb_str, "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3b82f6"}})   # bait  = blue
view.setStyle({"chain": "C"}, {"cartoon": {"color": "#f97316"}})   # target = orange
# highlight the surface we will redesign, in red
view.setStyle({"chain": "A", "resi": REDESIGN_REGION}, {"cartoon": {"color": "#ef4444"}})
view.zoomTo()
view.show()

print("Recipe (contig):", CONTIG)
print("We keep the bait's core, redesign the red surface, and keep the target fixed.")


## Step 1 — RFDiffusion: invent a new shape 🏗️

**RFDiffusion** is a generative AI for protein *shapes*. Starting from a cloud of random
noise, it gradually "denoises" it into a sensible protein backbone — a bit like an image
generator, but for 3D molecular structures. We tell it: *keep the bait's core and the target
where they are, but sculpt a fresh surface that wraps around the target.*

The next cell installs RFDiffusion and downloads its AI model. **This is the slowest step
(several minutes) — run it and wait.**


In [ ]:
# --- Install RFDiffusion + ColabDesign (this takes a few minutes) -------------
import os, time, sys

if not os.path.isdir("params"):
    os.makedirs("params", exist_ok=True)
    os.system("apt-get install aria2 -y")
    # download RFDiffusion's model weights + noise schedules in the background
    os.system("(aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; "
              "aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; "
              "aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; "
              "touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
    print("installing RFdiffusion ...")
    os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
    os.system("pip -q install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    os.system("pip -q install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    os.system("pip -q install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    os.system("pip -q install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
    os.system("cd RFdiffusion/env/SE3Transformer; pip -q install .")

if not os.path.isdir("colabdesign"):
    print("installing ColabDesign ...")
    os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
    print("waiting for model weights to finish downloading ...")
    os.makedirs("RFdiffusion/models", exist_ok=True)
    for m in ["Base_ckpt.pt", "Complex_base_ckpt.pt"]:
        while os.path.isfile(f"{m}.aria2"):
            time.sleep(5)
    os.system("mv Base_ckpt.pt Complex_base_ckpt.pt RFdiffusion/models")
    os.system("unzip -o schedules.zip; rm -f schedules.zip")

if "RFdiffusion" not in sys.path:
    os.environ["DGLBACKEND"] = "pytorch"
    sys.path.append("RFdiffusion")

print("✅ RFDiffusion ready.")


The cell below defines the plumbing that actually runs RFDiffusion (adapted from the
ColabDesign project). You don't need to read it — just run it.


In [ ]:
# --- RFDiffusion runner (plumbing; adapted from sokrypton/ColabDesign) --------
import signal
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
import py3Dmol

from inference.utils import parse_pdb
from colabdesign.rf.utils import get_ca, fix_contigs, fix_pdb
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

def _run(command, steps, num_designs=1):
    # Run RFDiffusion and show a live progress bar + snapshot of the shape.
    def start(command):
        os.system(f"nohup {command} > rfdiff.log 2>&1 & echo $! > /dev/shm/pid")
        return int(open("/dev/shm/pid").read().strip())
    def alive(pid):
        try: os.kill(pid, 0)
        except OSError: return False
        return True

    out = widgets.Output()
    bar = widgets.FloatProgress(min=0, max=1, description="running", bar_style="info")
    display(widgets.VBox([bar, out]))
    for n in range(steps):
        if os.path.isfile(f"/dev/shm/{n}.pdb"): os.remove(f"/dev/shm/{n}.pdb")
    pid = start(command)
    fail = False
    for _ in range(num_designs):
        for n in range(steps):
            wait = True
            while wait and not fail:
                time.sleep(0.1)
                if os.path.isfile(f"/dev/shm/{n}.pdb"):
                    s = open(f"/dev/shm/{n}.pdb").read()
                    if s[-3:] == "TER": wait = False
                    elif not alive(pid): fail = True
                elif not alive(pid): fail = True
            if fail:
                bar.bar_style, bar.description = "danger", "failed"; break
            bar.value = (n + 1) / steps
            with out:
                out.clear_output(wait=True)
                xyz, b = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                fig = plt.figure(); fig.set_dpi(90); fig.set_figwidth(5); fig.set_figheight(5)
                ax = fig.add_subplot(111); ax.set_xticks([]); ax.set_yticks([])
                plot_pseudo_3D(xyz, c=b, cmin=0.5, cmax=0.9, ax=ax); plt.show()
            if os.path.exists(f"/dev/shm/{n}.pdb"): os.remove(f"/dev/shm/{n}.pdb")
        if fail: break
    while alive(pid): time.sleep(0.1)

def run_diffusion(contigs, path, pdb, iterations=50, hotspot="", num_designs=1):
    full = f"outputs/{path}"; os.makedirs(full, exist_ok=True)
    opts = [f"inference.output_prefix={full}", f"inference.num_designs={num_designs}",
            f"diffuser.T={iterations}"]
    contigs = contigs.replace(",", " ").split()
    fixed_chains = [seg[0] for c in contigs for seg in c.split("/") if seg[:1].isalpha()]
    pdb_str = pdb_to_string(pdb, chains=sorted(set(fixed_chains)))
    with open(f"{full}/input.pdb", "w") as fh: fh.write(pdb_str)
    parsed = parse_pdb(f"{full}/input.pdb")
    opts.append(f"inference.input_pdb={full}/input.pdb")
    contigs = fix_contigs(contigs, parsed)
    if hotspot: opts.append(f"ppi.hotspot_res=[{hotspot}]")
    opts.append(f"'contigmap.contigs=[{' '.join(contigs)}]'")
    opts += ["inference.dump_pdb=True", "inference.dump_pdb_path='/dev/shm'"]
    cmd = f"./RFdiffusion/run_inference.py {' '.join(opts)}"
    print("contigs:", contigs)
    _run(cmd, iterations, num_designs)
    for n in range(num_designs):
        for f in [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
                  f"outputs/traj/{path}_{n}_Xt-1_traj.pdb", f"{full}_{n}.pdb"]:
            with open(f) as h: s = h.read()
            with open(f, "w") as h: h.write(fix_pdb(s, contigs))
    return contigs

print("✅ Runner defined.")


Now we actually run it. To keep the demo quick we make **one** new backbone with a modest
number of denoising steps. As it runs you'll watch the shape emerge from noise. In a real
campaign we generate **dozens** of these and keep the best.


In [ ]:
contigs = run_diffusion(CONTIG, path="demo", pdb=INPUT_COMPLEX, iterations=50, num_designs=1)
print("\n✅ RFDiffusion made a new backbone: outputs/demo_0.pdb")


In [ ]:
# --- Look at the freshly generated backbone (grey) hugging the target (orange) ---
with open("outputs/demo_0.pdb") as fh:
    new_bb = fh.read()

view = py3Dmol.view(width=760, height=520)
view.addModel(new_bb, "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "#94a3b8"}})   # new bait backbone
view.setStyle({"chain": "B"}, {"cartoon": {"color": "#f97316"}})   # target
view.zoomTo(); view.show()
print("Grey = the AI's freshly invented backbone. Orange = the fixed target.")
print("It has no amino-acid sequence yet — that's the next step.")


## Following one design all the way through 🎯

RFDiffusion is **random** — every run gives a different shape, and most won't end up binding.
So instead of depending on the single backbone we just made, we'll now follow **one real
backbone from a full pipeline run — "design 28"** — which we already know leads to a working
binder. Everything from here uses that design.

Here it is: the redesigned bait backbone (chain A) around the target (chain B).


In [ ]:
with open(DESIGN_BACKBONE) as fh:
    d28 = fh.read()

view = py3Dmol.view(width=760, height=520)
view.addModel(d28, "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3b82f6"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "#f97316"}})
view.zoomTo(); view.show()
print("Design 28: a new bait backbone (blue) shaped to wrap around the target (orange).")


## Step 2 — ProteinMPNN: choose the sequence ✍️

RFDiffusion gave us a *shape* but no *sequence* — like a sculpture with no material. **ProteinMPNN**
looks at the 3D backbone and picks the chain of amino acids most likely to fold into exactly
that shape.

We only let it design the **new** positions (the surface we resurfaced). Everything we chose
to keep — the bait's core and the whole target — is **frozen** to its original sequence.


In [ ]:
import numpy as np
from colabdesign.mpnn import mk_mpnn_model

# Positions ProteinMPNN is allowed to design (0-based, within the bait chain A).
# These are exactly the "invented" stretches from our recipe: a 12-residue insert
# and a 6-residue tail. Everything else is frozen.
DESIGN_POS = list(range(32, 44)) + list(range(71, 77))
NUM_SEQS = 2                       # how many sequences to propose
TEMP = 0.25                        # lower = more conservative choices

mpnn = mk_mpnn_model()
mpnn.prep_inputs(pdb_filename=DESIGN_BACKBONE, chain="A,B", rm_aa="C")  # no cysteines

# Freeze every position except DESIGN_POS by heavily biasing it toward its native residue.
S = mpnn._inputs["S"]              # native amino acid at each position
L = len(S)
bias = np.zeros((L, 20))
for i in range(L):
    if i not in DESIGN_POS:        # frozen position -> lock to what's already there
        bias[i, :] = -1e9
        bias[i, S[i]] = 0.0
mpnn._inputs["bias"] = bias

out = mpnn.sample(num=1, batch=NUM_SEQS, temperature=TEMP)

# Show what ProteinMPNN chose at the designed positions, vs. the original design-28 choice
REF_DESIGNED = "KVYAVPGSDKIT" + "GGMKRL"   # what the real pipeline picked for these positions
print(f"Designed positions: {len(DESIGN_POS)} residues on the resurfaced patch\n")
print(f"Original pipeline chose : {REF_DESIGNED}")
for k in range(NUM_SEQS):
    seq = out["seq"][k]
    receptor = seq.split("/")[0] if "/" in seq else seq[:77]
    picked = "".join(receptor[i] for i in DESIGN_POS)
    print(f"ProteinMPNN proposal {k+1}: {picked}   (confidence score {out['score'][k]:.2f})")

print("\nDifferent runs pick slightly different residues — protein design has many good answers.")


## Step 3 — Boltz-2: does it actually bind? 🔬

We now have a full designed protein (a new sequence on a new shape). But a design on paper
means nothing until we check it **folds and grabs the target**. **Boltz-2** is a structure-
prediction AI (a cousin of AlphaFold): give it two protein sequences and it predicts the 3D
structure of the pair — including *whether and how they stick together*.

We feed it the designed bait **sequence** (no structure — we don't want to give it the
answer) together with the target sequence, and see what it predicts. We use the exact
sequence the pipeline validated for design 28.

The next cell installs Boltz-2 and downloads its model (a few minutes the first time).


In [ ]:
!pip -q install boltz

# Read the designed bait + target sequences for design 28
seqs = {}
name = None
for line in open(DESIGN_FASTA):
    line = line.strip()
    if line.startswith(">"): name = line[1:]
    elif line: seqs[name] = seqs.get(name, "") + line

receptor_seq = seqs["receptor"]   # the designed bait
effector_seq = seqs["effector"]   # the target (unchanged)
print("Designed bait :", receptor_seq)
print("Target        :", effector_seq)


In [ ]:
# --- Write Boltz's input file: two protein chains, no evolutionary hints (single sequence) ---
import os
os.makedirs("boltz_run", exist_ok=True)
yaml_text = f'''version: 1
sequences:
  - protein:
      id: A
      sequence: {receptor_seq}
      msa: empty
  - protein:
      id: B
      sequence: {effector_seq}
      msa: empty
'''
with open("boltz_run/input.yaml", "w") as fh:
    fh.write(yaml_text)
print(yaml_text)


In [ ]:
# --- Run the prediction (downloads the model on first use; then a couple of minutes) -------
!boltz predict boltz_run/input.yaml --out_dir boltz_run/out --recycling_steps 3 --diffusion_samples 1 --output_format pdb --num_workers 0 --override
print("\n✅ Boltz-2 finished.")


## The verdict ✅

Below is Boltz-2's prediction of the designed bait (<span style="color:#3b82f6">blue</span>)
together with the target (<span style="color:#f97316">orange</span>). Because we gave Boltz
**only the sequences** and no structure, it had to work out the binding pose from scratch —
so if the two proteins end up locked together in the intended way, that's real evidence the
design works.

The **confidence scores** (ipTM / pTM, 0–1) are Boltz's own estimate of how sure it is. A high
**ipTM** in particular means it's confident about *how the two proteins fit together*.


In [ ]:
import glob, json, py3Dmol

pdbs = sorted(glob.glob("boltz_run/out/**/*.pdb", recursive=True))
jsons = sorted(glob.glob("boltz_run/out/**/*confidence*.json", recursive=True))
pred_pdb = open(pdbs[0]).read()

view = py3Dmol.view(width=760, height=520)
view.addModel(pred_pdb, "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3b82f6"}})   # designed bait
view.setStyle({"chain": "B"}, {"cartoon": {"color": "#f97316"}})   # target
view.zoomTo(); view.show()

if jsons:
    conf = json.load(open(jsons[0]))
    print("Boltz-2 confidence:")
    print(f"   overall (confidence_score): {conf.get('confidence_score', float('nan')):.2f}")
    print(f"   how-they-fit-together (ipTM): {conf.get('iptm', float('nan')):.2f}")
    print(f"   each-protein-folded  (pTM):   {conf.get('ptm', float('nan')):.2f}")

print("\nIn our full pipeline this design is graded 'tier A': Boltz predicts the bait binds")
print("the target in the intended pose on the very first try — no extra nudging needed.")


## Recap — and what the real pipeline adds 🧠

You just ran the **core loop** of the pipeline on one working example:

1. **RFDiffusion** invented a new backbone shape for the bait around the target.
2. **ProteinMPNN** chose an amino-acid sequence to realise that shape.
3. **Boltz-2** predicted the pair and confirmed the design **binds the target**. ✅

In a real project we don't stop at one design — we do a few extra things:

- **Scale up:** generate *dozens to hundreds* of backbones and sequences, because most designs
  *don't* work. The pipeline automatically keeps only the promising ones.
- **Steering (the tricky bit we skipped):** sometimes Boltz predicts the bait binding the
  target in the *wrong* place. The pipeline can *nudge* the design to fix the pose and re-check —
  we chose an example that didn't need this.
- **Second opinions:** promising designs are re-checked with **independent** tools (AlphaFold3,
  Rosetta energy calculations) so we don't trust a single AI.
- **The wet lab:** the very best designs are actually **made and tested on real plants** to see
  if they trigger the immune response.

**The big idea:** instead of waiting for evolution, we can now *design* proteins to recognise a
chosen target — and check them on a computer before ever touching a test tube. 🌱
